# 第15章：Split-K 并行 -- K维并行切分

## 前置知识
- 第09章：分块矩阵乘法基础
- 第14章：软件流水线
- 理解 GEMM 的标准并行策略 (M-N 维度并行)

## 学习目标
- 理解标准 GEMM 在 **tall-skinny 矩阵** 上并行度不足的问题
- 掌握 **Split-K** 算法的原理
- 实现 Split-K GEMM kernel (使用 atomic_add 进行部分和的归约)
- 了解 Split-K vs 标准 GEMM 的适用场景
- 在不同矩阵形状下 benchmark 两种策略

## 对应 CUDA 概念
- CUDA 中的 Split-K 通常用于 M,N 很小但 K 很大的场景
- 类似于对 reduction 维度进行并行化

In [1]:
import torch
import triton
import triton.language as tl

## 15.1 标准 GEMM 的并行度问题

### 标准并行策略回顾

在标准 GEMM 中，并行度 = M 方向 blocks x N 方向 blocks：

```
grid = (cdiv(M, BLOCK_M), cdiv(N, BLOCK_N))
total_programs = cdiv(M, BLOCK_M) * cdiv(N, BLOCK_N)
```

### 问题：tall-skinny 矩阵

```
Case 1: 方阵 (M=4096, N=4096, K=1024)
  BLOCK_M=128, BLOCK_N=128
  grid = (32, 32) = 1024 个 program
  → GPU (假设 80 SMs) 可以完全利用 ✓

Case 2: tall-skinny (M=128, N=128, K=8192)
  BLOCK_M=128, BLOCK_N=128
  grid = (1, 1) = 1 个 program!!
  → 只用了 1 个 SM, 79 个 SM 空闲! ✗

Case 3: 宽矩阵 (M=256, N=256, K=16384)
  BLOCK_M=128, BLOCK_N=128
  grid = (2, 2) = 4 个 program
  → 只用了 4 个 SM ✗
```

```
并行度不足的图解:

标准 GEMM (M=128, N=128, K=8192):

C 矩阵:          K 循环:
┌────┐            ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│ P0 │            │k0│k1│k2│k3│k4│k5│k6│k7│..│  │ ← P0 独自处理整个 K!
└────┘            └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘
只有 1 个            K 很长, 但只有 1 个 program 在做
program             其他 SM 全部空闲!
```

## 15.2 Split-K 算法

### 核心思想

将 K 维度切分为 SPLIT_K 份，每份由不同的 program 计算**部分和**，
最后通过 **原子加 (atomic_add)** 合并结果。

```
Split-K (SPLIT_K=4, M=128, N=128, K=8192):

K 维度被切成 4 份:
┌──────────┬──────────┬──────────┬──────────┐
│ K chunk 0│ K chunk 1│ K chunk 2│ K chunk 3│
│ k=0:2048 │k=2048:4096│k=4096:6144│k=6144:8192│
└──────────┴──────────┴──────────┴──────────┘
     ↓            ↓            ↓            ↓
  Program 0    Program 1    Program 2    Program 3
  (SM 0)       (SM 1)       (SM 2)       (SM 3)
     ↓            ↓            ↓            ↓
  C_partial_0  C_partial_1  C_partial_2  C_partial_3
     ↓            ↓            ↓            ↓
     └──────── atomic_add ────────────────┘
                     ↓
                C = Σ C_partial_i
```

### 并行度提升

```
标准 GEMM:  programs = cdiv(M,BM) * cdiv(N,BN)
Split-K:    programs = cdiv(M,BM) * cdiv(N,BN) * SPLIT_K

M=128, N=128, BM=BN=128, K=8192:
  标准:   1 * 1 = 1 program
  Split4: 1 * 1 * 4 = 4 programs
  Split8: 1 * 1 * 8 = 8 programs
  Split16: 1 * 1 * 16 = 16 programs
```

### Split-K vs Standard 的权衡

```
Split-K 优势:
  + 增加并行度 (更多 program)
  + 更好的 SM 利用率
  + 适合 M,N 小, K 大的场景

Split-K 劣势:
  - 需要 atomic_add (额外开销)
  - C 矩阵需要初始化为 0
  - atomic_add 可能产生竞争 (多个 program 写同一位置)
  - 当标准并行度已经足够时, 反而增加了开销
```

## 15.3 实现：Split-K GEMM

In [2]:
@triton.jit
def matmul_splitk_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    SPLIT_K: tl.constexpr,
):
    """
    Split-K GEMM kernel:
    每个 program 计算 K 维度的一个子区间的部分和,
    然后用 atomic_add 累加到 C 矩阵。
    
    grid = (cdiv(M, BM) * cdiv(N, BN), SPLIT_K)
    axis 0: M-N 并行
    axis 1: K 并行 (split)
    """
    # ========== pid 解码 ==========
    pid_mn = tl.program_id(0)  # M-N 维度的 program id
    pid_k = tl.program_id(1)   # K 维度的 split id
    
    grid_n = tl.cdiv(N, BLOCK_N)
    pid_m = pid_mn // grid_n
    pid_n = pid_mn % grid_n
    
    # ========== 计算当前 split 负责的 K 范围 ==========
    # 将 K 均匀分成 SPLIT_K 份
    k_per_split = tl.cdiv(K, SPLIT_K)  # 每个 split 的 K 大小
    k_start = pid_k * k_per_split       # 当前 split 的起始 K
    k_end = min(k_start + k_per_split, K)  # 当前 split 的结束 K
    
    # ========== Block Pointer ==========
    # A: 从第 k_start 列开始
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, k_start),
        block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    # B: 从第 k_start 行开始
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(k_start, pid_n * BLOCK_N),
        block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    # ========== 累加器 ==========
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    # ========== 在分配的 K 范围内循环 ==========
    # 注意: 只处理 [k_start, k_end) 这一段
    for k in range(k_start, k_end, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    # ========== 使用 atomic_add 写回部分和 ==========
    # 所有 SPLIT_K 个 program 都会写入 C 的同一个区域
    # 必须用 atomic_add 来避免 race condition
    c = acc.to(tl.float16)
    
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn
    c_mask = (rm[:, None] < M) & (rn[None, :] < N)
    
    # atomic_add: 多个 program 的部分和安全累加
    tl.atomic_add(c_ptrs, c, mask=c_mask)

In [3]:
def matmul_splitk(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, SPLIT_K=4):
    """
    Split-K GEMM 的 host 端包装函数。
    注意: C 必须初始化为 0, 因为使用了 atomic_add。
    """
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    
    # 必须初始化为 0! (atomic_add 会在此基础上累加)
    c = torch.zeros((M, N), device=a.device, dtype=a.dtype)
    
    # 2D grid: (M-N 并行, K split 并行)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N), SPLIT_K)
    
    matmul_splitk_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        SPLIT_K=SPLIT_K,
    )
    return c

In [4]:
# ========== 标准 GEMM (用于对比) ==========
@triton.jit
def matmul_standard_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N), block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, acc.to(tl.float16), boundary_check=(0, 1))

def matmul_standard(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    matmul_standard_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c

In [5]:
# ========== 正确性验证 ==========
torch.manual_seed(42)

print("Split-K 正确性验证:")
for M, N, K, SPLIT_K in [
    (128, 128, 2048, 4),
    (256, 256, 4096, 8),
    (512, 512, 8192, 4),
    (2048, 2048, 1024, 2),
]:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    c_splitk = matmul_splitk(a, b, SPLIT_K=SPLIT_K)
    c_ref = torch.matmul(a, b)
    
    max_err = (c_splitk - c_ref).abs().max().item()
    rel_err = torch.norm(c_splitk.float() - c_ref.float()) / torch.norm(c_ref.float())
    print(f"  ({M:>4}x{K:>5}) @ ({K:>5}x{N:>4}), SPLIT_K={SPLIT_K}: "
          f"max_err={max_err:.4f}, rel_err={rel_err:.6f}")

Split-K 正确性验证:
  ( 128x 2048) @ ( 2048x 128), SPLIT_K=4: max_err=0.1250, rel_err=0.000481
  ( 256x 4096) @ ( 4096x 256), SPLIT_K=8: max_err=0.2500, rel_err=0.000545
  ( 512x 8192) @ ( 8192x 512), SPLIT_K=4: max_err=0.2500, rel_err=0.000487
  (2048x 1024) @ ( 1024x2048), SPLIT_K=2: max_err=0.1250, rel_err=0.000358


## 15.4 Split-K 的图解分析

```
标准 GEMM vs Split-K:

标准 GEMM (M=128, N=128, K=8192):

  A (128 x 8192)          B (8192 x 128)         C (128 x 128)
  ┌────────────────────┐   ┌───┐                  ┌───┐
  │                    │   │   │                  │   │
  │   1 个 program     │ @ │   │      =           │   │
  │   处理全部 K       │   │   │                  │   │
  └────────────────────┘   │   │                  └───┘
                           │   │
  programs: 1              │   │
  SM利用率: 1.25%          └───┘


Split-K (SPLIT_K=8):

  A (128 x 8192)          B (8192 x 128)         C (128 x 128)
  ┌──────┬──────┬──────┐   ┌───┐                  ┌───┐
  │  P0  │  P1  │  P2  │   │P0 │                  │   │
  ├──────┼──────┼──────┤   ├───┤                  │ Σ │ ← atomic_add
  │  P3  │  P4  │  P5  │   │P1 │      →           │   │
  ├──────┼──────┼──────┤ @ ├───┤                  └───┘
  │  P6  │  P7  │      │   │P2 │
  └──────┴──────┴──────┘   ├───┤
                           │...│
  每个 program 只处理       ├───┤
  K/8 = 1024 的K           │P7 │
                           └───┘
  programs: 8
  SM利用率: 10%
```

In [6]:
# ========== Split-K vs Standard: 不同矩阵形状对比 ==========
def benchmark_fn(fn, a, b, num_warmup=10, num_rep=50, **kwargs):
    for _ in range(num_warmup):
        fn(a, b, **kwargs)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(num_rep):
        fn(a, b, **kwargs)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / num_rep

print("Standard vs Split-K: 不同矩阵形状")
print(f"{'Shape':>25} | {'#Programs':>10} {'Standard':>12} | {'SPLIT_K':>8} {'#Programs':>10} {'Split-K':>12} | {'加速':>6}")
print("-" * 100)

BM, BN, BK = 128, 128, 32

test_cases = [
    # (M, N, K, SPLIT_K)
    (128,  128,  8192,  8),   # 极端 tall-skinny
    (256,  256,  8192,  8),   # tall-skinny
    (256,  256,  4096,  4),   # tall-skinny
    (512,  512,  4096,  4),   # 中等
    (1024, 1024, 4096,  4),   # 接近方阵
    (2048, 2048, 1024,  2),   # 标准方阵
    (4096, 4096, 1024,  2),   # 大方阵
]

for M, N, K, SPLIT_K in test_cases:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    progs_std = triton.cdiv(M, BM) * triton.cdiv(N, BN)
    progs_sk = progs_std * SPLIT_K
    
    try:
        ms_std = benchmark_fn(matmul_standard, a, b)
        ms_sk = benchmark_fn(matmul_splitk, a, b, SPLIT_K=SPLIT_K)
        speedup = ms_std / ms_sk
        shape_str = f"({M}x{K})@({K}x{N})"
        print(f"{shape_str:>25} | {progs_std:>10} {ms_std:>11.3f}ms | {SPLIT_K:>8} {progs_sk:>10} {ms_sk:>11.3f}ms | {speedup:>5.2f}x")
    except Exception as e:
        print(f"{M:>6}x{N:>6}x{K:>6}: ERROR {str(e)[:40]}")

Standard vs Split-K: 不同矩阵形状
                    Shape |  #Programs     Standard |  SPLIT_K  #Programs      Split-K |     加速
----------------------------------------------------------------------------------------------------
    (128x8192)@(8192x128) |          1       0.131ms |        8          8       0.040ms |  3.31x
    (256x8192)@(8192x256) |          4       0.122ms |        8         32       0.041ms |  3.00x
    (256x4096)@(4096x256) |          4       0.067ms |        4         16       0.041ms |  1.65x
    (512x4096)@(4096x512) |         16       0.064ms |        4         64       0.040ms |  1.61x
  (1024x4096)@(4096x1024) |         64       0.071ms |        4        256       0.047ms |  1.50x
  (2048x1024)@(1024x2048) |        256       0.033ms |        2        512       0.054ms |  0.62x
  (4096x1024)@(1024x4096) |       1024       0.103ms |        2       2048       0.157ms |  0.65x


## 15.5 何时使用 Split-K？

### 决策规则

```
计算标准 GEMM 的 program 数:
  standard_programs = cdiv(M, BM) * cdiv(N, BN)

如果 standard_programs < num_SMs * 2:
  → GPU 利用率不足, 考虑 Split-K
  → SPLIT_K ≈ ceil(num_SMs * 2 / standard_programs)

如果 standard_programs >= num_SMs * 2:
  → 标准 GEMM 已经有足够并行度
  → Split-K 的 atomic_add 开销可能得不偿失
```

### 典型场景

```
适合 Split-K 的场景:
  - Transformer 中的 attention: Q @ K^T (seq_len 较小, d_model 较大)
  - MLP 的某些层: 输入 batch 小, hidden_dim 大
  - 特征提取: 通道数 (K) 远大于空间尺寸 (M, N)

不适合 Split-K 的场景:
  - 大方阵 (M, N 都大)
  - K 很小 (Split 之后每个 chunk 太小, 没有足够计算量)
  - 对精度要求极高 (atomic_add 的非确定性顺序可能影响浮点结果)
```

In [7]:
# ========== 不同 SPLIT_K 值的效果 ==========
M, N, K = 256, 256, 8192
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

print(f"矩阵: ({M}x{K}) @ ({K}x{N})")
print(f"标准 programs: {triton.cdiv(M,128) * triton.cdiv(N,128)}")
print(f"\n{'SPLIT_K':>10} | {'Programs':>10} | {'时间(ms)':>10} | {'TFLOPS':>8}")
print("-" * 50)

flops = 2.0 * M * N * K

# 标准版
ms_std = benchmark_fn(matmul_standard, a, b)
tflops_std = flops / (ms_std * 1e-3) / 1e12
print(f"{'标准':>10} | {triton.cdiv(M,128)*triton.cdiv(N,128):>10} | {ms_std:>10.3f} | {tflops_std:>8.2f}")

# Split-K 不同值
for sk in [2, 4, 8, 16, 32]:
    progs = triton.cdiv(M, 128) * triton.cdiv(N, 128) * sk
    try:
        ms = benchmark_fn(matmul_splitk, a, b, SPLIT_K=sk)
        tflops = flops / (ms * 1e-3) / 1e12
        print(f"{sk:>10} | {progs:>10} | {ms:>10.3f} | {tflops:>8.2f}")
    except Exception as e:
        print(f"{sk:>10} | {progs:>10} | {'FAIL':>10} | {str(e)[:20]}")

矩阵: (256x8192) @ (8192x256)
标准 programs: 4

   SPLIT_K |   Programs |     时间(ms) |   TFLOPS
--------------------------------------------------
        标准 |          4 |      0.124 |     8.63
         2 |          8 |      0.124 |     8.67
         4 |         16 |      0.066 |    16.19
         8 |         32 |      0.041 |    26.43
        16 |         64 |      0.032 |    33.18
        32 |        128 |      0.034 |    31.58


In [8]:
# ========== Split-K vs cuBLAS ==========
print("\nSplit-K vs Standard vs cuBLAS")
print(f"{'Shape':>25} | {'Standard':>10} {'Split-K':>10} {'cuBLAS':>10} | {'SK/cuBLAS':>10}")
print("-" * 80)

for M, N, K, SK in [
    (128,  128,  8192,  16),
    (256,  256,  4096,  8),
    (512,  512,  4096,  4),
    (2048, 2048, 2048,  2),
]:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    ms_std = benchmark_fn(matmul_standard, a, b)
    ms_sk = benchmark_fn(matmul_splitk, a, b, SPLIT_K=SK)
    
    # cuBLAS
    for _ in range(10):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(50):
        torch.matmul(a, b)
    e.record()
    torch.cuda.synchronize()
    ms_cu = s.elapsed_time(e) / 50
    
    shape_str = f"({M}x{K})@({K}x{N})"
    print(f"{shape_str:>25} | {ms_std:>9.3f}ms {ms_sk:>9.3f}ms {ms_cu:>9.3f}ms | {ms_cu/ms_sk:>9.2f}x")


Split-K vs Standard vs cuBLAS
                    Shape |   Standard    Split-K     cuBLAS |  SK/cuBLAS
--------------------------------------------------------------------------------
    (128x8192)@(8192x128) |     0.128ms     0.032ms     0.015ms |      0.48x
    (256x4096)@(4096x256) |     0.064ms     0.033ms     0.015ms |      0.47x
    (512x4096)@(4096x512) |     0.064ms     0.040ms     0.018ms |      0.47x
  (2048x2048)@(2048x2048) |     0.061ms     0.094ms     0.060ms |      0.64x


## 15.6 总结

### 本章要点

1. **并行度不足问题**：当 M,N 较小时，标准 GEMM 的 grid 太小，无法充分利用 GPU

2. **Split-K 算法**：
   - 将 K 维度切成 SPLIT_K 份
   - 每份由不同的 program 计算部分和
   - 使用 atomic_add 合并结果
   - 并行度从 `cdiv(M,BM)*cdiv(N,BN)` 提升到 `cdiv(M,BM)*cdiv(N,BN)*SPLIT_K`

3. **使用场景**：
   - 适合：M,N 小但 K 大的 tall-skinny 矩阵
   - 不适合：大方阵 (标准 GEMM 已有足够并行度)
   - 决策依据：standard_programs < num_SMs * 2 时考虑 Split-K

4. **实现注意事项**：
   - C 必须初始化为 0 (atomic_add 累加)
   - atomic_add 有一定性能开销
   - SPLIT_K 不宜过大 (每个 split 的计算量太少会导致 overhead 主导)

### 练习

1. **FP32 累加**：修改 Split-K kernel，使用 FP32 输出 + FP32 atomic_add，对比精度
2. **两阶段 Split-K**：不使用 atomic_add，而是先写到临时 buffer，再用另一个 kernel 做 reduce
3. **自动选择**：编写一个函数，根据 (M, N, K, num_SMs) 自动决定是否使用 Split-K 以及最优 SPLIT_K 值
4. **思考题**：为什么 Split-K 使用 atomic_add 而不是先写到各自的 buffer 再 reduce？（提示：内存效率）

### 下一章预告

第16章将深入 **混合精度策略**，探讨 FP16/FP32/TF32/BF16 在 GEMM 中的使用方式和权衡。